# Notebook 01: Real data: download files and build the sample sheet

**Goal.** Download the public GEO files and create **data/sample_sheet.csv**. **Skip this notebook in the dry run** (it needs internet and your choices).

This notebook is run once per cohort. Everything downloaded goes to data/raw/ and is never uploaded to GitHub.

> REAL DATA ONLY - skip this notebook in the dry run.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** Switches the project from simulated to real data. All real outputs go to separate folders (data/processed/real, results/real), so simulated results never mix with real ones.

In [ ]:
set_profile("real")      # then: Kernel > Restart, and re-run the first cell

**Cell 3.** Asks NCBI GEO which supplementary files the series offers. This removes the need to hunt for download buttons.

*You should see:* A list of file names such as GSE121654_RAW.tar.

In [ ]:
from mlc.dataset import geo_list, geo_download, geo_sample_table
url, files = geo_list("GSE121654", "suppl")      # also try GSE129788
for f in files: print(f)

**Cell 4.** Downloads one file to data/raw/GSE121654/. If it already exists it is skipped, so the cell is safe to re-run. Large files can take several minutes.

In [ ]:
name = "GSE121654_RAW.tar"                     # EDIT: choose from the list above
geo_download(url + name, RAW / "GSE121654" / name)

**Cell 5.** Unpacks the archive and prints every file with its size and what kind of file it looks like (10x matrix, genes, barcodes, dense table, h5ad). Use **inventory** whenever a loader complains: it shows what you actually have.

In [ ]:
import tarfile
tarfile.open(RAW / "GSE121654" / name).extractall(RAW / "GSE121654")
from mlc.dataset import inventory
inventory(RAW / "GSE121654")

**Cell 6.** Downloads each series' sample table (title, age, sex, tissue as written by the authors) and saves it as CSV. Open the CSV in Excel; you will copy its information into the sample sheet.

In [ ]:
for acc in ["GSE121654", "GSE129788"]:
    t = geo_sample_table(acc, RAW / "geo_meta")
    t.to_csv(RAW / f"{acc}_geo_samples.csv", index=False)
    print(acc, t.shape); print(t.head(3).T)

**Cell 7.** Creates an empty sample sheet with the right column names. Fill one row per **independent biological sample**. **files** = folder (or file) holding that sample's counts; **prefix** = the file-name start when several samples share one folder (for example GSM5416205_6mth). **eligible_primary** = TRUE only for adult, control, independent microglia samples.

In [ ]:
template = pd.DataFrame(columns=["cohort", "sample_id", "gsm", "species", "tissue", "region", "age_months", "sex",
                                 "condition", "replicate_unit", "eligible_primary", "files", "prefix"])
template.to_csv(DATA / "sample_sheet.csv", index=False)
print("Now open data/sample_sheet.csv in Excel and fill ONE ROW PER MOUSE/DONOR (see the table in the guide).")

**Cell 8.** Checks the inclusion rule (at least 3 young and 3 old samples per cohort) and prints a plain-language verdict per cohort. Cohorts that fail are not used for the main tests. It also records which cohorts the next notebooks should process.

*You should see:* e.g. 'GSE121654: 12 eligible samples; young=4 mid=0 old=8 -> OK'

In [ ]:
import json
from mlc.dataset import validate_sheet
ok = validate_sheet(pd.read_csv(SHEET))
(DATA / "cohorts_real.json").write_text(json.dumps({"mouse": sorted(ok.cohort.unique()), "human_sn": []}))